## 6.0 Improved Model — Transfer Learning with MobileNetV2 (*Member 2*)

**Author of sections 6.0 – 7.0: Ek (Member 2).**

The baseline CNN (Section 5) learns every filter from 240 training images. The
improved model instead **re-uses visual features learned on 1.28 million ImageNet
photographs** (transfer learning) and only has to learn how to map them to our three
classes. This is the pairing the brief lists as acceptable: *custom CNN vs transfer
learning*.

**Rules that keep the comparison fair** (from Member 1's handover):

| Rule | How this section follows it |
|---|---|
| Same images | `train_df`, `val_df`, `test_df` come from Member 1's saved manifests; nothing is re-split |
| Same preprocessing | `make_dataset()` (decode → 224 × 224 → ÷255 → batch) is reused unchanged |
| Same augmentation | `make_augmentation()` (flip, rotation 0.1, zoom 0.1) is the first layer of the model |
| Same seed | `tf.keras.utils.set_random_seed(SEED)` before every model is built |
| Validation only | Every tuning and selection decision uses `val_df`; **the test set is not touched in 6.0 – 7.0**. Member 3 evaluates the final model on it once, in 8.0 |

The first cell below works in two situations: in the **master notebook** it simply
re-uses Member 1's objects; in a **separate copy** (or on the offline demo machine) it
rebuilds exactly the same objects from `data/split_manifests/`.

In [ ]:
# Author: Ek (Member 2)
# Cell 6.0 - Member 2 Setup: Reuse Member 1's Split and Pipeline

import os
import json
import time
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

# In the master notebook, Member 1's cells 1.0 - 5.2 have already defined these objects.
REQUIRED = ["train_df", "val_df", "test_df", "make_dataset", "make_augmentation",
            "CLASS_NAMES", "LABEL_MAP", "SPLIT_PATH", "RESULTS_PATH", "FIGURES_PATH", "SEED"]
STANDALONE = not all(name in globals() for name in REQUIRED)

if STANDALONE:
    # ---- Member 2's own copy / offline demo machine: rebuild the same objects ----
    SEED = 42
    tf.keras.utils.set_random_seed(SEED)

    # Colab path by default; set the environment variable IT8414_PROJECT_DIR to run elsewhere
    PROJECT_DIR = Path(os.environ.get("IT8414_PROJECT_DIR", "/content/IT8414_Tomato_Project"))

    # Nothing to upload: if the split manifests are missing, download the project from GitHub
    if not (PROJECT_DIR / "data" / "split_manifests" / "train_split.csv").exists() and not PROJECT_DIR.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "--branch", os.environ.get("IT8414_REPO_BRANCH", "main"),
                        "https://github.com/eman-271/it8414-tomato-leaf-disease-classification.git",
                        str(PROJECT_DIR)], check=False)

    # If Member 1's share ZIP was uploaded to /content, unzip it once
    if Path("/content").exists() and not PROJECT_DIR.exists():
        share_zips = sorted(Path("/content").glob("*Member1_Share*.zip"))
        if share_zips:
            with zipfile.ZipFile(share_zips[0]) as z:
                z.extractall("/content")
            print("Share package unzipped from:", share_zips[0].name)

    SELECTED_PATH = PROJECT_DIR / "data" / "selected"
    SPLIT_PATH = PROJECT_DIR / "data" / "split_manifests"
    RESULTS_PATH = PROJECT_DIR / "results"
    FIGURES_PATH = RESULTS_PATH / "figures"
    FIGURES_PATH.mkdir(parents=True, exist_ok=True)

    CLASS_NAMES = ["Healthy", "Early Blight", "Late Blight"]                       # label IDs 0, 1, 2
    EXPECTED_CLASSES = ["Tomato___healthy", "Tomato___Early_blight", "Tomato___Late_blight"]
    LABEL_MAP = {folder: i for i, folder in enumerate(EXPECTED_CLASSES)}
    IMG_SIZE = (224, 224)
    BATCH_SIZE = 16
    AUTOTUNE = tf.data.AUTOTUNE

    # ---- The four functions below are copied UNCHANGED from Member 1 (Cell 4.0) ----
    def load_manifest(csv_path):
        # Rebuild the full image path from the class label and file name
        frame = pd.read_csv(csv_path)
        if "filename" not in frame.columns:
            frame["filename"] = frame["image_path"].apply(lambda p: Path(p).name)
        frame["image_path"] = [str(SELECTED_PATH / label / name)
                               for label, name in zip(frame["label"], frame["filename"])]
        frame["label_id"] = frame["label"].map(LABEL_MAP)
        return frame[["image_path", "label", "label_id"]].reset_index(drop=True)

    def load_and_preprocess(image_path, label):
        # Decode RGB -> resize to 224 x 224 -> scale pixels to 0-1
        image = tf.io.read_file(image_path)
        image = tf.io.decode_image(image, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMG_SIZE)
        image = tf.cast(image, tf.float32) / 255.0
        return image, label

    def make_dataset(df, training=False, batch_size=BATCH_SIZE):
        dataset = tf.data.Dataset.from_tensor_slices((df["image_path"].values, df["label_id"].values))
        dataset = dataset.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
        dataset = dataset.cache()
        if training:   # shuffle the training data only
            dataset = dataset.shuffle(buffer_size=len(df), seed=SEED, reshuffle_each_iteration=True)
        return dataset.batch(batch_size).prefetch(AUTOTUNE)

    def make_augmentation():
        return tf.keras.Sequential([
            tf.keras.layers.RandomFlip("horizontal"),
            tf.keras.layers.RandomRotation(0.1),
            tf.keras.layers.RandomZoom(0.1)
        ], name="data_augmentation")
    # ---- end of the code copied from Member 1 ----

    train_df = load_manifest(SPLIT_PATH / "train_split.csv")
    val_df = load_manifest(SPLIT_PATH / "validation_split.csv")
    test_df = load_manifest(SPLIT_PATH / "test_split.csv")
    print("Standalone mode: split loaded from", SPLIT_PATH)
else:
    print("Master-notebook mode: re-using Member 1's split, make_dataset() and make_augmentation().")

print(f"{len(train_df)} training / {len(val_df)} validation / {len(test_df)} test images")

# Stop early, with instructions, if this is not the official split of the report (240 / 30 / 30)
if (len(train_df), len(val_df), len(test_df)) != (240, 30, 30) or globals().get("SPLIT_LOADED") is False:
    raise RuntimeError(
        f"This is not the official 80/10/10 split (found {len(train_df)} / {len(val_df)} / {len(test_df)} images"
        + (", made new in this session" if globals().get("SPLIT_LOADED") is False else "") + ").\n"
        "Fix: Runtime -> Run all again. Cell 1.0 downloads the official split from GitHub, and Cell 4.0 then prints "
        "'Saved split loaded (ratio 80/10/10): 240 training / 30 validation / 30 test images'.")
print("TensorFlow", tf.__version__, "| Keras", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "")
GPU_NAME = "T4 GPU" if tf.config.list_physical_devices("GPU") else "CPU"
print("Hardware:", GPU_NAME)

### 6.0 Data check before training (Member 2's verification of the pipeline)

The task allocation asks Member 2 to verify the preprocessed data shape and the batch
loaders before using them. The next cell checks, and stops with an error if anything
is wrong:

1. the split in memory is **identical to the saved manifests** (so nobody re-split by accident);
2. the split sizes and the **stratification** (80 / 10 / 10 images per class);
3. **no image appears in two splits**, and every file exists;
4. one batch has shape *(batch, 224, 224, 3)*, type float32 and pixel range 0 – 1;
5. the validation loader is **not shuffled**, so predictions line up with `val_df` (needed by Member 3);
6. after the MobileNetV2 rescaling layer the pixels are in the range −1 to 1.

In [ ]:
# Author: Ek (Member 2)
# Cell 6.0 - Data Check: Manifests, Stratification, Leakage and Batch Shapes

checks = []
frames = {"train": train_df, "validation": val_df, "test": test_df}
manifests = {s: pd.read_csv(SPLIT_PATH / f"{s}_split.csv") for s in frames}

def keys(frame):
    # class folder + file name identifies an image uniquely
    return set(Path(p).parent.name + "/" + Path(p).name for p in frame["image_path"])

for s, frame in frames.items():
    on_disk = set(manifests[s]["label"] + "/" + manifests[s]["filename"])
    checks.append((f"{s}: identical to the saved manifest", keys(frame) == on_disk))
    checks.append((f"{s}: every image file exists", all(Path(p).exists() for p in frame["image_path"])))

expected = {"train": 80, "validation": 10, "test": 10}
for s, frame in frames.items():
    counts = frame["label_id"].value_counts().to_dict()
    checks.append((f"{s}: {expected[s]} images per class {counts}",
                   counts == {0: expected[s], 1: expected[s], 2: expected[s]}))

checks.append(("no image shared by train / validation",
               not keys(train_df) & keys(val_df)))
checks.append(("no image shared by train / test", not keys(train_df) & keys(test_df)))
checks.append(("no image shared by validation / test", not keys(val_df) & keys(test_df)))

images, labels = next(iter(make_dataset(train_df, training=True, batch_size=8)))
checks.append((f"training batch shape {tuple(images.shape)}", tuple(images.shape) == (8, 224, 224, 3)))
checks.append((f"pixel dtype {images.dtype.name}, range {float(tf.reduce_min(images)):.3f} - "
               f"{float(tf.reduce_max(images)):.3f}",
               images.dtype == tf.float32 and float(tf.reduce_min(images)) >= 0
               and float(tf.reduce_max(images)) <= 1))

val_labels = np.concatenate([y.numpy() for _, y in make_dataset(val_df, batch_size=8)])
checks.append(("validation loader keeps the order of val_df", np.array_equal(val_labels, val_df["label_id"].values)))

rescaled = tf.keras.layers.Rescaling(2.0, offset=-1.0)(images)
checks.append((f"MobileNetV2 input range {float(tf.reduce_min(rescaled)):.3f} - {float(tf.reduce_max(rescaled)):.3f}",
               float(tf.reduce_min(rescaled)) >= -1 and float(tf.reduce_max(rescaled)) <= 1))

for description, passed in checks:
    print(("PASS  " if passed else "FAIL  ") + description)
assert all(passed for _, passed in checks), "Data check failed - do not train until this is fixed (see the FAIL lines above)."
print(f"\nAll {len(checks)} checks passed.")

### 6.0 Overall workflow (Figure 4.1)

The diagram shows how both models share one data pipeline, how each is tuned on the
validation set only, and where the single test-set evaluation (Member 3, Section 8.0)
happens. It is Figure 4.1 in the report.

In [ ]:
# Author: Ek (Member 2)
# Cell 6.0 - Overall Workflow Diagram (Figure 4.1)

from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(12, 7.2))
ax.set_xlim(0, 12)
ax.set_ylim(-0.35, 7.2)
ax.axis("off")

def box(x, y, w, h, title, detail, colour):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.03", fc=colour, ec="#334155"))
    ax.text(x + w / 2, y + h * 0.64, title, ha="center", va="center", fontsize=10.5, weight="bold")
    ax.text(x + w / 2, y + h * 0.28, detail, ha="center", va="center", fontsize=8.5)

def arrow(x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", color="#334155"))

blue, green, amber, red = "#dbeafe", "#dcfce7", "#fef3c7", "#fee2e2"
box(3.5, 6.2, 5, 0.8, "PlantVillage tomato images (300)", "balanced 100 per class, quality-checked (3.0-4.0)", blue)
box(3.5, 5.0, 5, 0.8, "Stratified 80/10/10 split, saved manifests", "240 train / 30 validation / 30 test; leakage checks (5.4)", blue)
box(3.5, 3.8, 5, 0.8, "Shared preprocessing (tf.data)", "decode RGB -> 224x224 -> /255; training-only augmentation", blue)
arrow(6, 6.2, 6, 5.8); arrow(6, 5.0, 6, 4.6)

box(0.3, 2.2, 5.2, 1.0, "Baseline: custom CNN from scratch", "3 conv blocks, 101,699 params; tuned batch / lr / kernel (5.0)", green)
box(6.5, 2.2, 5.2, 1.0, "Improved: MobileNetV2 transfer learning", "stage 1 head only, stage 2 fine-tuning; tuned lr / batch /\ndropout / fine-tune depth / fine-tune lr (6.0-7.0)", green)
arrow(5, 3.8, 2.9, 3.2); arrow(7, 3.8, 9.1, 3.2)

box(3.5, 1.0, 5, 0.8, "Model selection on the VALIDATION set only", "lowest validation loss; early stopping restores the best epoch", amber)
arrow(2.9, 2.2, 4.5, 1.8); arrow(9.1, 2.2, 7.5, 1.8)

box(3.5, -0.1, 5, 0.8, "One-time TEST evaluation (Member 3, 8.0-9.0)", "accuracy, macro P/R/F1, ROC-AUC, confusion matrix, Grad-CAM", red)
arrow(6, 1.0, 6, 0.7)
ax.set_title("Overall Workflow: Shared Data Pipeline, Two Models, Validation-Only Tuning, One Test Evaluation", fontsize=12)
plt.savefig(FIGURES_PATH / "fig4_1_overall_workflow.png", dpi=150, bbox_inches="tight")
plt.show()

### 6.1 Architecture

**Why MobileNetV2?** (full justification in report Section 2.4)

- **Transfer learning suits 240 training images.** Filters learned on ImageNet
  (edges, colours, textures, shapes) are general, so only the classifier on top has to
  be learned from our data (Yosinski et al., 2014).
- **Small and fast.** MobileNetV2 has about 2.26 million parameters, roughly
  one-eleventh of ResNet50 (about 23.6 million without its top layer). It was designed
  for phones (Sandler et al., 2018), which fits a screening tool used by farmers in the field.
- **Same input size.** It expects 224 × 224 images, which Member 1's pipeline already produces.

**How MobileNetV2 processes an image.** It is built from 17 *inverted residual
bottleneck* blocks. Each block (1) **expands** the channels with a 1 × 1 convolution,
(2) filters each channel separately with a 3 × 3 **depthwise** convolution, and
(3) **projects** back to few channels with a 1 × 1 convolution that has *no* activation
(a *linear bottleneck*, so information is not destroyed by ReLU in a low-dimensional
space). A shortcut adds the block's input to its output when the shapes match.
Activations are ReLU6, and every convolution is followed by batch normalisation.
For a 224 × 224 image, the last layer (`out_relu`) gives **7 × 7 × 1,280** feature maps.

**Our model (layer by layer):**

| Layer | Output | Purpose |
|---|---|---|
| `data_augmentation` | 224 × 224 × 3 | Member 1's training-only flip / rotation / zoom |
| `rescale_to_minus1_1` | 224 × 224 × 3 | MobileNetV2 was trained on pixels in −1 … 1; Member 1's pipeline gives 0 … 1, so *x* × 2 − 1 |
| `mobilenetv2_1.00_224` | 7 × 7 × 1,280 | Pre-trained feature extractor (frozen in stage 1) |
| `global_avg_pool` | 1,280 | One value per feature map (same idea as the baseline) |
| `head_dropout` | 1,280 | Dropout (rate tuned) — overfitting control |
| `output_softmax` | 3 | One probability per class |

The base is always called with `training=False`, so its **batch-normalisation layers
stay in inference mode** even when some of its layers are unfrozen for fine-tuning.
Otherwise the moving averages learned on ImageNet would be overwritten by statistics
from batches of only 8 – 32 leaf images (Chollet, 2020).

In [ ]:
# Author: Ek (Member 2)
# Cell 6.1 - Improved Model Architecture (MobileNetV2 transfer learning)

IMPROVED_INPUT_SHAPE = (224, 224, 3)
BASE_NAME = "mobilenetv2_1.00_224"     # Keras' name for the MobileNetV2 base


def build_improved_model(dropout_rate=0.2, augment=True):
    # Pre-trained base without its ImageNet classifier (include_top=False)
    base = tf.keras.applications.MobileNetV2(
        input_shape=IMPROVED_INPUT_SHAPE, include_top=False, weights="imagenet")
    base.trainable = False                                    # stage 1: feature extraction

    inputs = tf.keras.Input(shape=IMPROVED_INPUT_SHAPE, name="image_0_to_1")
    x = make_augmentation()(inputs) if augment else inputs   # training only (Member 1's block)
    x = tf.keras.layers.Rescaling(2.0, offset=-1.0, name="rescale_to_minus1_1")(x)   # 0..1 -> -1..1
    x = base(x, training=False)                               # keep batch norm in inference mode
    x = tf.keras.layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = tf.keras.layers.Dropout(dropout_rate, name="head_dropout")(x)
    outputs = tf.keras.layers.Dense(3, activation="softmax", name="output_softmax")(x)
    return tf.keras.Model(inputs, outputs, name="improved_mobilenetv2")


def set_fine_tuning(model, from_block=None):
    # Unfreeze the base from 'block_<from_block>_expand' to the end; None = keep it frozen.
    # Batch-normalisation layers always stay frozen.
    base = model.get_layer(BASE_NAME)
    if from_block is None:
        base.trainable = False
        return 0
    base.trainable = True
    start = [layer.name for layer in base.layers].index(f"block_{from_block}_expand")
    for i, layer in enumerate(base.layers):
        layer.trainable = i >= start and not isinstance(layer, tf.keras.layers.BatchNormalization)
    return len(base.layers) - start


def count_params(weights):
    return int(sum(np.prod(w.shape) for w in weights))


tf.keras.utils.set_random_seed(SEED)
preview_model = build_improved_model()
preview_model.summary()

# Parameter counts for report Table 4.4 (stage 1 = frozen base; stage 2 = fine-tuned from block 14)
base_preview = preview_model.get_layer(BASE_NAME)
layer_rows = []
for layer in preview_model.layers:
    layer_rows.append({"Layer": layer.name, "Type": layer.__class__.__name__,
                       "Output shape": str(tuple(layer.output.shape)),
                       "Parameters": count_params(layer.weights)})
layer_table = pd.DataFrame(layer_rows)
display(layer_table)
print("Total parameters:", f"{preview_model.count_params():,}")
print("Trainable in stage 1 (head only):", f"{count_params(preview_model.trainable_weights):,}")
for block in [14, 11]:
    n_layers = set_fine_tuning(preview_model, block)
    print(f"Trainable when fine-tuning from block {block} ({n_layers} base layers unfrozen, BN frozen):",
          f"{count_params(preview_model.trainable_weights):,}")
set_fine_tuning(preview_model, None)
print("Last convolutional output (for Grad-CAM):", base_preview.output.shape, "from layer 'out_relu'")
layer_table.to_csv(RESULTS_PATH / "improved_layer_table.csv", index=False)

### 6.2 Training strategy and helper

Training follows the standard two-stage transfer-learning recipe (Chollet, 2020):

1. **Stage 1 – feature extraction.** The base is frozen; only the dropout + softmax head
   (3,843 weights) is trained with Adam. A frozen base cannot overfit, and the randomly
   initialised head does not send large, random gradients into the pre-trained filters.
2. **Stage 2 – fine-tuning.** Starting from the best stage-1 weights, the last blocks of
   the base are unfrozen and trained with a **10 – 100 × smaller learning rate**, so the
   ImageNet filters are adapted to leaf lesions rather than overwritten.

| Setting | Value | Same as baseline? |
|---|---|---|
| Loss | Sparse categorical cross-entropy | Yes |
| Optimiser | Adam (learning rate tuned per stage) | Yes |
| Early stopping | `val_loss`, patience 5, best weights restored | Yes (maximum 50 epochs in stage 1 as for the baseline, 20 in stage 2) |
| Augmentation | Member 1's `make_augmentation()` | Yes |
| Extra regularisation | Dropout before the classifier (rate tuned); frozen BN; small fine-tuning learning rate | New |
| Seed | 42, reset before every model | Yes |

`train_improved()` returns a summary with **the same columns as Member 1's
`run_experiment()`**, so Tables 5.1 and 5.4 line up, plus `Stage` and `Trainable params`.

**Re-running without changing the numbers.** GPU training is not bit-for-bit
deterministic, so a second "Run all" can move a validation loss in the third or fourth
decimal. Every run is therefore saved to `results/improved_runs.json`. Set
`M2_REUSE_SAVED = True` before the final *Restart session and run all* of the merged
notebook, and these cells re-load the saved runs and models instead of retraining, so the
notebook outputs match the report exactly.

In [ ]:
# Author: Ek (Member 2)
# Cell 6.2 - Training Helper for the Improved Model

IMPROVED_MAX_EPOCHS = 50      # stage 1 (head only), same limit as the baseline
FINE_TUNE_MAX_EPOCHS = 20     # stage 2 (fine-tuning)
IMPROVED_PATIENCE = 5

# False while experimenting; True for the final merged "Run all" (re-uses saved runs)
M2_REUSE_SAVED = False

RUNS_FILE = RESULTS_PATH / "improved_runs.json"
STAGE1_WEIGHTS = RESULTS_PATH / "improved_stage1_selected.weights.h5"
FINAL_MODEL_FILE = RESULTS_PATH / "improved_mobilenetv2_final.keras"
saved_runs = json.loads(RUNS_FILE.read_text()) if (M2_REUSE_SAVED and RUNS_FILE.exists()) else {}
if M2_REUSE_SAVED:
    print(f"M2_REUSE_SAVED = True: {len(saved_runs)} saved runs loaded from {RUNS_FILE.name}")

if "KeepBestWeights" not in globals():
    # Same callback as Member 1's Cell 5.2 (needed when this section runs on its own)
    class KeepBestWeights(tf.keras.callbacks.Callback):
        def on_train_begin(self, logs=None):
            self.best_val_loss, self.best_weights, self.best_epoch = np.inf, None, 0

        def on_epoch_end(self, epoch, logs=None):
            if logs["val_loss"] < self.best_val_loss:
                self.best_val_loss, self.best_weights, self.best_epoch = logs["val_loss"], self.model.get_weights(), epoch + 1

        def on_train_end(self, logs=None):
            if self.best_weights is not None:
                self.model.set_weights(self.best_weights)


def train_improved(exp, parameter, value, stage=1, learning_rate=1e-3, batch_size=16,
                   dropout_rate=0.2, fine_tune_from=None, init_weights=None,
                   augment=True, early_stopping=True, max_epochs=None, verbose=2):
    """Trains one fresh, seeded improved model and returns (summary, history, model)."""
    if M2_REUSE_SAVED and str(exp) in saved_runs:
        print(f"Exp. {exp}: saved result re-used (no retraining)")
        run = saved_runs[str(exp)]
        return run["summary"], run["history"], None

    max_epochs = max_epochs or (IMPROVED_MAX_EPOCHS if stage == 1 else FINE_TUNE_MAX_EPOCHS)
    tf.keras.utils.set_random_seed(SEED)                     # same start for every experiment
    exp_train_ds = make_dataset(train_df, training=True, batch_size=batch_size)
    exp_val_ds = make_dataset(val_df, batch_size=batch_size)

    model = build_improved_model(dropout_rate=dropout_rate, augment=augment)
    if init_weights is not None:
        model.load_weights(init_weights)                     # stage 2 starts from stage 1
    set_fine_tuning(model, fine_tune_from)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                  metrics=["accuracy"])

    keep_best = KeepBestWeights()
    callbacks = ([tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=IMPROVED_PATIENCE,
                                                   restore_best_weights=True), keep_best]
                 if early_stopping else [])

    start = time.time()
    run_history = model.fit(exp_train_ds, validation_data=exp_val_ds, epochs=max_epochs,
                            callbacks=callbacks, verbose=verbose)
    train_time = time.time() - start

    h = {k: [float(v) for v in vals] for k, vals in run_history.history.items()}
    b = keep_best.best_epoch - 1 if early_stopping else len(h["loss"]) - 1   # reported epoch
    summary = {
        "Exp.": exp, "Parameter varied": parameter, "Value": value,
        "Epochs run": len(h["loss"]), "Best epoch": b + 1,
        "Train acc.": round(h["accuracy"][b], 4), "Val. acc.": round(h["val_accuracy"][b], 4),
        "Train loss": round(h["loss"][b], 4), "Val. loss": round(h["val_loss"][b], 4),
        "Params": model.count_params(), "Train time (min)": round(train_time / 60, 1),
        "Stage": stage, "Trainable params": count_params(model.trainable_weights),
    }
    print(summary)

    saved_runs[str(exp)] = {"summary": summary, "history": h}
    RUNS_FILE.write_text(json.dumps(saved_runs, indent=1))
    return summary, h, model


# Containers for all improved-model experiments
improved_results, improved_histories, improved_models, improved_configs = [], {}, {}, {}

## 7.0 Hyperparameter Experiments and Overfitting Control — *Member 2*

### 7.1 Stage 1: head-only experiments (one factor at a time)

As in Member 1's baseline tuning, **one hyperparameter is changed at a time** from a
reference, and each is tested in both directions. Early stopping chooses the number of
epochs for every run.

| Exp. | Parameter varied | Value | Everything else | Why |
|---|---|---|---|---|
| I1 | Reference | – | lr 0.001, batch 16, dropout 0.2 | Common starting point for a new head (Chollet, 2020) |
| I2 | Learning rate | 0.0001 | as reference | Smaller steps: slower but smoother |
| I3 | Learning rate | 0.01 | as reference | Larger steps: a linear head on fixed features may tolerate them |
| I4 | Batch size | 8 | as reference | Best batch for the baseline; more, noisier updates |
| I5 | Batch size | 32 | as reference | Fewer, smoother updates |
| I6 | Dropout rate | 0.0 | as reference | No dropout: is it needed? |
| I7 | Dropout rate | 0.5 | as reference | Strong dropout (Srivastava et al., 2014) |

**Selection rule (fixed before running):** lowest validation loss at the best epoch,
exactly as for the baseline. Loss is used rather than accuracy, because one of the 30
validation images moves accuracy by 3.3 percentage points.

In [ ]:
# Author: Ek (Member 2)
# Cell 7.1 - Stage 1 Experiments I1 - I7 (frozen MobileNetV2, validation set only)

IMPROVED_REFERENCE = {"learning_rate": 1e-3, "batch_size": 16, "dropout_rate": 0.2}

STAGE1_PLAN = [
    ("I1", "Reference", "lr 0.001 / bs 16 / dropout 0.2", {}),
    ("I2", "Learning rate", "0.0001", {"learning_rate": 1e-4}),
    ("I3", "Learning rate", "0.01", {"learning_rate": 1e-2}),
    ("I4", "Batch size", "8", {"batch_size": 8}),
    ("I5", "Batch size", "32", {"batch_size": 32}),
    ("I6", "Dropout rate", "0.0", {"dropout_rate": 0.0}),
    ("I7", "Dropout rate", "0.5", {"dropout_rate": 0.5}),
]

for exp, parameter, value, changes in STAGE1_PLAN:
    config = {**IMPROVED_REFERENCE, **changes}
    print(f"\n=== Exp. {exp}: {parameter} = {value} ===")
    summary, history, model = train_improved(exp, parameter, value, stage=1, **config)
    improved_results.append(summary)
    improved_histories[exp] = history
    improved_models[exp] = model
    improved_configs[exp] = config

stage1_df = pd.DataFrame(improved_results)
STAGE1_SELECTED = stage1_df.loc[stage1_df["Val. loss"].idxmin(), "Exp."]
STAGE1_CONFIG = improved_configs[STAGE1_SELECTED]
print(f"\nStage 1 selected: Exp. {STAGE1_SELECTED} {STAGE1_CONFIG} "
      f"(validation loss {stage1_df['Val. loss'].min():.4f})")

# Save the selected stage-1 weights: stage 2 starts from them
if improved_models[STAGE1_SELECTED] is not None:
    improved_models[STAGE1_SELECTED].save_weights(str(STAGE1_WEIGHTS))
print("Stage-1 weights:", STAGE1_WEIGHTS.name, "exists:", STAGE1_WEIGHTS.exists())

### 7.2 Stage 2: fine-tuning experiments

Starting from the selected stage-1 weights (same batch size and dropout), the last part
of MobileNetV2 is unfrozen. Two more hyperparameters are tuned:

| Exp. | Fine-tuned from | Learning rate | Trainable weights | Why |
|---|---|---|---|---|
| I8 | block 14 (blocks 14 – 16 + last 1 × 1 conv) | 0.00001 | ≈ 1.5 M | Adapt only the most class-specific (7 × 7) features |
| I9 | block 11 (blocks 11 – 16 + last conv) | 0.00001 | ≈ 1.9 M | Deeper adaptation, more overfitting risk |
| I10 | block 14 | 0.0001 | ≈ 1.5 M | 10 × larger step: faster, but may destroy pre-trained features |

The earlier blocks (edges, colours, simple textures) are general and stay frozen
(Yosinski et al., 2014). If no fine-tuning run beats stage 1 on validation loss, the
stage-1 model is kept — fine-tuning is only used if the evidence supports it.

In [ ]:
# Author: Ek (Member 2)
# Cell 7.2 - Stage 2 Fine-Tuning Experiments I8 - I10 (validation set only)

FINE_TUNE_PLAN = [
    ("I8", "Fine-tune depth", "from block 14, lr 1e-5", {"fine_tune_from": 14, "learning_rate": 1e-5}),
    ("I9", "Fine-tune depth", "from block 11, lr 1e-5", {"fine_tune_from": 11, "learning_rate": 1e-5}),
    ("I10", "Fine-tune learning rate", "from block 14, lr 1e-4", {"fine_tune_from": 14, "learning_rate": 1e-4}),
]

for exp, parameter, value, changes in FINE_TUNE_PLAN:
    config = {"batch_size": STAGE1_CONFIG["batch_size"], "dropout_rate": STAGE1_CONFIG["dropout_rate"], **changes}
    print(f"\n=== Exp. {exp}: {parameter} = {value} (starts from Exp. {STAGE1_SELECTED}) ===")
    summary, history, model = train_improved(exp, parameter, value, stage=2,
                                             init_weights=str(STAGE1_WEIGHTS), **config)
    improved_results.append(summary)
    improved_histories[exp] = history
    improved_models[exp] = model
    improved_configs[exp] = config

### 7.3 Results table and final selection

All ten runs in one table (report Table 5.2). The final improved model is the run with
the **lowest validation loss**: either the selected stage-1 model or one of the
fine-tuned models. The chart is Figure D.2 in the report appendix.

In [ ]:
# Author: Ek (Member 2)
# Cell 7.3 - Improved-Model Results Table, Validation-Loss Chart and Selection

improved_df = pd.DataFrame(improved_results)
improved_df.to_csv(RESULTS_PATH / "improved_hyperparameter_results.csv", index=False)
display(improved_df)

# Figure D.2: validation loss per epoch for every experiment (dot = best epoch)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for exp, history in improved_histories.items():
    ax = axes[0] if exp in dict((e, 1) for e, *_ in STAGE1_PLAN) else axes[1]
    losses = history["val_loss"]
    line, = ax.plot(range(1, len(losses) + 1), losses, label=f"{exp}")
    best = int(np.argmin(losses))
    ax.scatter(best + 1, losses[best], color=line.get_color(), zorder=3)
axes[0].set_title("Stage 1 (frozen base): validation loss")
axes[1].set_title(f"Stage 2 (fine-tuning from Exp. {STAGE1_SELECTED}): validation loss")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.grid(True)
    ax.legend()
axes[0].set_ylabel("Validation loss")
plt.tight_layout()
plt.savefig(FIGURES_PATH / "figD_2_improved_experiment_val_loss.png", dpi=150, bbox_inches="tight")
plt.show()

# Final selection: lowest validation loss over the stage-1 winner and the fine-tuning runs
candidates = improved_df[(improved_df["Exp."] == STAGE1_SELECTED) | (improved_df["Stage"] == 2)]
final_row = candidates.loc[candidates["Val. loss"].idxmin()]
FINAL_EXP = final_row["Exp."]
FINAL_CONFIG = improved_configs[FINAL_EXP]
print(f"Final improved model: Exp. {FINAL_EXP} ({final_row['Parameter varied']} = {final_row['Value']})")
print(f"Best epoch {final_row['Best epoch']}, validation loss {final_row['Val. loss']}, "
      f"validation accuracy {final_row['Val. acc.']:.2%}")
print("Decision based on the validation set only; the test set was not used.")

### 7.4 Overfitting control: does the regularisation actually help?

To *demonstrate* (not just claim) that the regularisation works, the final configuration
is trained a second time with **all overfitting controls switched off**: no augmentation,
no dropout and no early stopping (the full epoch budget, final-epoch weights kept). The
two-stage schedule, learning rates and batch size stay the same. Comparing the two
learning curves shows what the controls do (report Figure 5.3 and Table 5.3).

In [ ]:
# Author: Ek (Member 2)
# Cell 7.4 - Overfitting-Control Ablation (validation set only)

final_is_fine_tuned = FINAL_CONFIG.get("fine_tune_from") is not None
STAGE1_UNREG_WEIGHTS = RESULTS_PATH / "improved_ablation_stage1.weights.h5"

# A1: stage 1 without augmentation, dropout or early stopping
abl1, abl1_history, abl1_model = train_improved(
    "A1", "No regularisation (stage 1)", "no aug / dropout 0 / no early stop", stage=1,
    learning_rate=STAGE1_CONFIG["learning_rate"], batch_size=STAGE1_CONFIG["batch_size"],
    dropout_rate=0.0, augment=False, early_stopping=False, max_epochs=IMPROVED_MAX_EPOCHS, verbose=0)
ablation_rows, ablation_history = [abl1], {k: list(v) for k, v in abl1_history.items()}

if final_is_fine_tuned:
    if abl1_model is not None:
        abl1_model.save_weights(str(STAGE1_UNREG_WEIGHTS))
    # A2: the same fine-tuning as the final model, again without any regularisation
    abl2, abl2_history, _ = train_improved(
        "A2", "No regularisation (stage 2)", "no aug / dropout 0 / no early stop", stage=2,
        learning_rate=FINAL_CONFIG["learning_rate"], batch_size=FINAL_CONFIG["batch_size"],
        dropout_rate=0.0, fine_tune_from=FINAL_CONFIG["fine_tune_from"],
        init_weights=str(STAGE1_UNREG_WEIGHTS), augment=False, early_stopping=False,
        max_epochs=FINE_TUNE_MAX_EPOCHS, verbose=0)
    ablation_rows.append(abl2)
    ablation_history = {k: ablation_history[k] + list(abl2_history[k]) for k in ablation_history}


def stitched_history(final_exp):
    # Stage-1 history (up to its restored best epoch) followed by the stage-2 history
    h1 = improved_histories[STAGE1_SELECTED]
    if improved_configs[final_exp].get("fine_tune_from") is None:
        return {k: list(v) for k, v in h1.items()}, None
    keep = int(improved_df.loc[improved_df["Exp."] == STAGE1_SELECTED, "Best epoch"].iloc[0])
    h2 = improved_histories[final_exp]
    return {k: list(h1[k][:keep]) + list(h2[k]) for k in h1}, keep


reg_history, ft_start = stitched_history(FINAL_EXP)
unreg_switch = len(abl1_history["loss"]) if final_is_fine_tuned else None


def gap_row(name, h, reported_epoch):
    i = reported_epoch - 1
    return {"Model": name, "Reported epoch": reported_epoch,
            "Train loss": round(h["loss"][i], 4), "Val. loss": round(h["val_loss"][i], 4),
            "Gap (val - train)": round(h["val_loss"][i] - h["loss"][i], 4),
            "Train acc.": round(h["accuracy"][i], 4), "Val. acc.": round(h["val_accuracy"][i], 4),
            "Lowest val. loss (epoch)": f"{min(h['val_loss']):.4f} ({int(np.argmin(h['val_loss'])) + 1})"}

final_epoch_overall = ((ft_start or 0) + int(final_row["Best epoch"])) if final_is_fine_tuned else int(final_row["Best epoch"])
ablation_table = pd.DataFrame([
    gap_row("Regularised (final model: augmentation + dropout + early stopping)", reg_history, final_epoch_overall),
    gap_row("No regularisation (no augmentation, dropout 0, no early stopping)", ablation_history,
            len(ablation_history["loss"])),
])
ablation_table.to_csv(RESULTS_PATH / "improved_overfitting_ablation.csv", index=False)
display(ablation_table)

# Figure 5.3: training vs validation loss, regularised vs not
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, h, title, switch in [(axes[0], reg_history, "Regularised (final improved model)", ft_start),
                             (axes[1], ablation_history, "No regularisation", unreg_switch)]:
    epochs = range(1, len(h["loss"]) + 1)
    ax.plot(epochs, h["loss"], label="Training loss")
    ax.plot(epochs, h["val_loss"], label="Validation loss")
    if switch:
        ax.axvline(switch + 0.5, color="grey", linestyle="--", label="Fine-tuning starts")
    ax.set_title(title)
    ax.set_xlabel("Epoch (stage 1, then stage 2)")
    ax.grid(True)
    ax.legend()
axes[0].set_ylabel("Sparse categorical cross-entropy")
fig.suptitle("Overfitting Control: Training and Validation Loss With and Without Regularisation")
plt.tight_layout()
plt.savefig(FIGURES_PATH / "fig5_3_overfitting_control.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.5 Final improved model: learning curves and saved model

The final model (weights of its best epoch restored) is saved as
`results/improved_mobilenetv2_final.keras` for Member 3, who evaluates it **once** on the
test set in Section 8.0. The learning curves are Figure 5.2 in the report.

In [ ]:
# Author: Ek (Member 2)
# Cell 7.5 - Final Improved Model: Learning Curves, Validation Check and Save

if improved_models.get(FINAL_EXP) is not None:
    improved_model = improved_models[FINAL_EXP]
    improved_model.save(str(FINAL_MODEL_FILE))
else:
    improved_model = tf.keras.models.load_model(str(FINAL_MODEL_FILE))   # M2_REUSE_SAVED = True
print("Final improved model:", FINAL_MODEL_FILE)

# Figure 5.2: accuracy and loss over both training stages
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(reg_history["loss"]) + 1)
for ax, metric, title in [(axes[0], "accuracy", "Accuracy"), (axes[1], "loss", "Loss")]:
    ax.plot(epochs, reg_history[metric], label=f"Training {title.lower()}")
    ax.plot(epochs, reg_history[f"val_{metric}"], label=f"Validation {title.lower()}")
    if ft_start:
        ax.axvline(ft_start + 0.5, color="grey", linestyle=":", label=f"Fine-tuning starts (after epoch {ft_start})")
    ax.axvline(final_epoch_overall, color="black", linestyle="--", label=f"Restored best epoch ({final_epoch_overall})")
    ax.set_title(f"Improved MobileNetV2 - Training and Validation {title}")
    ax.set_xlabel("Epoch (stage 1, then stage 2)")
    ax.set_ylabel(title)
    ax.grid(True)
    ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_PATH / "fig5_2_improved_learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Check: the saved model reproduces the best-epoch validation loss
val_loss_check, val_acc_check = improved_model.evaluate(make_dataset(val_df), verbose=0)
print(f"Validation loss {val_loss_check:.4f} | validation accuracy {val_acc_check:.2%} "
      f"(table: {final_row['Val. loss']} / {final_row['Val. acc.']:.2%})")

### 7.6 Activation hook for Grad-CAM (support for Member 3)

The task allocation asks Member 2 to help set up access to the model's activations.
In this model MobileNetV2 is a *nested* model, so its last convolutional output
(`out_relu`, 7 × 7 × 1,280) cannot be read with a simple `model.get_layer("out_relu")`
from the outer model. `make_activation_model()` re-connects the trained layers so that one
call returns **both** the last feature maps and the class probabilities — exactly what
Grad-CAM needs. Augmentation is skipped because it is inactive at prediction time
anyway. The check below uses a **validation** image; the test set stays untouched.

In [ ]:
# Author: Ek (Member 2)
# Cell 7.6 - Activation Model for Grad-CAM (Member 3 uses it in 9.0)

def make_activation_model(model):
    # Returns a model: image (0-1) -> [out_relu feature maps (7x7x1280), class probabilities]
    inputs = tf.keras.Input(shape=IMPROVED_INPUT_SHAPE)
    x = model.get_layer("rescale_to_minus1_1")(inputs)
    feature_maps = model.get_layer(BASE_NAME)(x, training=False)    # output of 'out_relu'
    y = model.get_layer("global_avg_pool")(feature_maps)
    y = model.get_layer("head_dropout")(y, training=False)
    probabilities = model.get_layer("output_softmax")(y)
    return tf.keras.Model(inputs, [feature_maps, probabilities], name="improved_activations")


activation_model = make_activation_model(improved_model)
sample_image, sample_label = next(iter(make_dataset(val_df.iloc[:1], batch_size=1)))
maps, probs = activation_model(sample_image)
direct = improved_model(sample_image, training=False)
print("Feature maps:", tuple(maps.shape), "| probabilities:", np.round(probs.numpy()[0], 4))
print("Same prediction as the full model:", np.allclose(probs.numpy(), direct.numpy(), atol=1e-5))
print("True class of this validation image:", CLASS_NAMES[int(sample_label[0])])

### 7.7 Member 2 summary, saved numbers and handover to Member 3

The next cell compares the two final models **on the validation set** (the test-set
comparison is Member 3's job in 8.0), and writes every number used in report Sections
4 and 5.1 – 5.2 to `results/M2_NUMBERS_FOR_REPORT.txt` and
`results/improved_validation_results.json`, in the same style as Member 1's file.

In [ ]:
# Author: Ek (Member 2)
# Cell 7.7 - Final Numbers and Ready-to-Paste Text (report Sections 4, 5.1, 5.2)

pct = lambda x: f"{x * 100:.2f}%"
lines = []
out = lines.append

baseline_json = RESULTS_PATH / "baseline_test_results.json"
baseline_info = json.loads(baseline_json.read_text()) if baseline_json.exists() else {}
base_val_acc = baseline_info.get("validation_accuracy", float("nan"))
base_val_loss = baseline_info.get("validation_loss", float("nan"))

tmp = build_improved_model()
total_params = tmp.count_params()
head_params = count_params(tmp.trainable_weights)
set_fine_tuning(tmp, 14)
ft14_params = count_params(tmp.trainable_weights)
set_fine_tuning(tmp, 11)
ft11_params = count_params(tmp.trainable_weights)
del tmp

ref = improved_df[improved_df["Exp."] == "I1"].iloc[0]
s1 = improved_df[improved_df["Exp."] == STAGE1_SELECTED].iloc[0]
fin = final_row
fin_cfg = FINAL_CONFIG
lr_stage1 = STAGE1_CONFIG["learning_rate"]
total_time = improved_df["Train time (min)"].sum()
final_time = float(s1["Train time (min)"]) + (float(fin["Train time (min)"]) if fin["Stage"] == 2 else 0.0)

out("=== M2-A. REPORT TABLE 4.4 (improved model parameters) ===")
out(f"Total parameters: {total_params:,} | trainable in stage 1 (head): {head_params:,} | "
    f"trainable when fine-tuning from block 14: {ft14_params:,} | from block 11: {ft11_params:,}")

out("")
out("=== M2-B. REPORT TABLE 4.2 (training configuration of the final improved model) ===")
out(f"Stage 1: Adam lr {lr_stage1:g}, batch {STAGE1_CONFIG['batch_size']}, dropout {STAGE1_CONFIG['dropout_rate']}, "
    f"early stopping (max {IMPROVED_MAX_EPOCHS}, patience {IMPROVED_PATIENCE}); best epoch {s1['Best epoch']} "
    f"(stopped at {s1['Epochs run']})")
if fin["Stage"] == 2:
    out(f"Stage 2: fine-tuned from block {fin_cfg['fine_tune_from']}, Adam lr {fin_cfg['learning_rate']:g}, "
        f"early stopping (max {FINE_TUNE_MAX_EPOCHS}, patience {IMPROVED_PATIENCE}); best epoch {fin['Best epoch']} "
        f"(stopped at {fin['Epochs run']}); trainable weights {int(fin['Trainable params']):,}")
else:
    out("Stage 2: not used - no fine-tuning run had a lower validation loss than stage 1")
out(f"Hardware: {GPU_NAME} | Final model training time: {final_time:.1f} min | all {len(improved_df)} runs: {total_time:.1f} min")

out("")
out("=== M2-C. REPORT TABLE 5.2 (Exp | Stage | Parameter | Value | Best epoch (of run) | Train acc | Val acc | Val loss | Trainable params) ===")
for _, r in improved_df.iterrows():
    out(f"{r['Exp.']} | {r['Stage']} | {r['Parameter varied']} | {r['Value']} | {r['Best epoch']} ({r['Epochs run']}) | "
        f"{pct(r['Train acc.'])} | {pct(r['Val. acc.'])} | {r['Val. loss']:.4f} | {int(r['Trainable params']):,}")

out("")
out("=== M2-D. TEXT for report 5.2 (selection) ===")
out(f"Stage 1 selected Exp. {STAGE1_SELECTED} ({s1['Parameter varied']} = {s1['Value']}) with validation loss "
    f"{s1['Val. loss']:.4f} (reference I1: {ref['Val. loss']:.4f}). The final improved model is Exp. {FINAL_EXP} "
    f"({fin['Parameter varied']} = {fin['Value']}): validation loss {fin['Val. loss']:.4f}, validation accuracy "
    f"{pct(fin['Val. acc.'])} at best epoch {fin['Best epoch']}. Baseline (final): validation loss {base_val_loss:.4f}, "
    f"validation accuracy {pct(base_val_acc)}. Decisions used the validation set only.")

out("")
out("=== M2-E. TEXT for report 5.1 (Figure 5.2, learning curves) ===")
h = reg_history
i = final_epoch_overall - 1
out(f"Stage 1 started at validation loss {h['val_loss'][0]:.4f} and validation accuracy {pct(h['val_accuracy'][0])} after one epoch. "
    f"At the restored best epoch ({final_epoch_overall} overall) training loss was {h['loss'][i]:.4f} and validation loss "
    f"{h['val_loss'][i]:.4f}; training accuracy {pct(h['accuracy'][i])}, validation accuracy {pct(h['val_accuracy'][i])}.")

out("")
out("=== M2-F. REPORT TABLE 5.3 (overfitting-control ablation) ===")
for _, r in ablation_table.iterrows():
    out(f"{r['Model']} | epoch {r['Reported epoch']} | train loss {r['Train loss']:.4f} | val loss {r['Val. loss']:.4f} | "
        f"gap {r['Gap (val - train)']:.4f} | train acc {pct(r['Train acc.'])} | val acc {pct(r['Val. acc.'])} | "
        f"lowest val loss {r['Lowest val. loss (epoch)']}")

out("")
out("=== M2-G. VALIDATION comparison (test-set comparison = Member 3, Cell 8.0) ===")
out(f"Baseline CNN: val acc {pct(base_val_acc)}, val loss {base_val_loss:.4f}, 101,699 params | "
    f"Improved MobileNetV2: val acc {pct(fin['Val. acc.'])}, val loss {fin['Val. loss']:.4f}, {total_params:,} params")

report_text = "\n".join(lines)
print(report_text)
(RESULTS_PATH / "M2_NUMBERS_FOR_REPORT.txt").write_text(report_text)

improved_info = {
    "final_experiment": str(FINAL_EXP), "stage1_selected": str(STAGE1_SELECTED),
    "final_config": {k: (float(v) if isinstance(v, (int, float)) and not isinstance(v, bool) else v)
                     for k, v in FINAL_CONFIG.items()},
    "validation_accuracy": float(fin["Val. acc."]), "validation_loss": float(fin["Val. loss"]),
    "best_epoch_stage": int(fin["Best epoch"]), "best_epoch_overall": int(final_epoch_overall),
    "parameters_total": int(total_params), "trainable_parameters_final": int(fin["Trainable params"]),
    "train_time_min": round(final_time, 1), "hardware": GPU_NAME,
    "model_file": FINAL_MODEL_FILE.name,
    "grad_cam_layer": "out_relu inside mobilenetv2_1.00_224 (use make_activation_model in Cell 7.6)",
    "input": "224 x 224 x 3, pixels 0-1 (rescaling to -1..1 is inside the model)",
}
(RESULTS_PATH / "improved_validation_results.json").write_text(json.dumps(improved_info, indent=2))
print("\nSaved results/M2_NUMBERS_FOR_REPORT.txt and results/improved_validation_results.json")

### 7.7 Handover to Member 3

| Name | What it is |
|---|---|
| `improved_model` | Final improved model (best epoch restored); also `results/improved_mobilenetv2_final.keras` |
| `make_activation_model(model)` | Returns `[out_relu feature maps, probabilities]` for Grad-CAM (Cell 7.6) |
| `improved_df` | All improved-model experiments (report Table 5.2) |
| `FINAL_EXP`, `FINAL_CONFIG` | Which run was selected and its settings |

- The model takes the **same 0 – 1 images** as the baseline, so `predict_leaf(path, improved_model)`
  and the evaluation code in Cell 5.8 work unchanged.
- The test set has **not** been used in 6.0 – 7.0. Evaluate `improved_model` on `test_df` **once**, in 8.0.
- Figures added: 4.1, 5.2, 5.3 and D.2. Tables added: 2.1, 2.2, 4.4 – 4.6, 5.2, 5.3 (Table 4.2 now covers both models). Member 3 continues from Figure 5.4 and Table 5.4.

In [ ]:
# Author: Ek (Member 2)
# Cell 7.7 - Download Member 2's New Results (Colab storage is temporary)

new_files = ["improved_mobilenetv2_final.keras", "improved_hyperparameter_results.csv", "improved_runs.json",
             "improved_overfitting_ablation.csv", "improved_layer_table.csv", "improved_validation_results.json",
             "M2_NUMBERS_FOR_REPORT.txt", "improved_stage1_selected.weights.h5",
             "figures/fig4_1_overall_workflow.png", "figures/fig5_2_improved_learning_curves.png",
             "figures/fig5_3_overfitting_control.png", "figures/figD_2_improved_experiment_val_loss.png"]
archive = RESULTS_PATH.parent / f"IT8414_Member2_results_{time.strftime('%Y-%m-%d')}.zip"
with zipfile.ZipFile(archive, "w") as z:
    for name in new_files:
        if (RESULTS_PATH / name).exists():
            z.write(RESULTS_PATH / name, f"IT8414_Tomato_Project/results/{name}")
print("ZIP:", archive, round(archive.stat().st_size / 1e6, 1), "MB")
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print("Not running in Colab - the ZIP stays in the project folder.")